In [1]:
"""Train the LODA detector and scale its anomaly scores."""

import os
import sys

# Clear any cached modules
if 'config' in sys.modules:
    del sys.modules['config']
if 'to_tflite' in sys.modules:
    del sys.modules['to_tflite']

import pandas as pd
from pyod.models.loda import LODA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import config
import to_tflite

# Build the detector from its configuration.
loda = LODA(
    contamination=config.CONTAMINATION,
    n_bins=config.N_BINS,
    n_random_cuts=config.N_RANDOM_CUTS,
)

# NOTE: all dataset features must be pre-normalized so that, after the int8
# quantization in keras/to_tflite.py, no value overflows the int8 range.
# Fit the configured features from the CSV.
features = pd.read_csv(config.CSV_PATH)[config.FEATURES].to_numpy()
X_train, X_test = train_test_split(
    features, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE
)
loda.fit(X_train)

# Scale the raw anomaly scores into the configured range.
raw_scores = loda.decision_function(X_train).reshape(-1, 1)
score_scaler = MinMaxScaler(feature_range=config.SCORE_RANGE).fit(raw_scores)
train_scores = score_scaler.transform(raw_scores).ravel()
test_scores = score_scaler.transform(
    loda.decision_function(X_test).reshape(-1, 1)
).ravel()

# Learn the scores with a small network and export .keras + .tflite.
to_tflite.to_tflite(
    name="loda",
    x_train=X_train,
    y_train=train_scores,
    x_test=X_test,
    y_test=test_scores,
    layers=config.LAYERS,
    learning_rate=config.LEARNING_RATE,
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    verbose=config.VERBOSE,
    feature_names=config.FEATURES,
)

2026-08-14 17:30:16.484784: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the nex

Epoch 1/50


2026-08-14 17:30:19.061006: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 776us/step - loss: 0.0023 - val_loss: 0.0017
Epoch 2/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 762us/step - loss: 0.0017 - val_loss: 0.0016
Epoch 3/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 750us/step - loss: 0.0016 - val_loss: 0.0016
Epoch 4/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 740us/step - loss: 0.0016 - val_loss: 0.0016
Epoch 5/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 772us/step - loss: 0.0016 - val_loss: 0.0015
Epoch 6/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 761us/step - loss: 0.0015 - val_loss: 0.0014
Epoch 7/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 773us/step - loss: 0.0014 - val_loss: 0.0014
Epoch 8/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 768us/step - loss: 0.0014 - val_loss: 0.0013
Epoch 9/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 777us/step - loss: 0.0013 - val_loss: 0.0013
Epoch 10/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 737us/step - loss: 0.0013 - val_loss: 0.0012
Epoch 11/50
3747/3747 ━━━━━━━━━━━━━━━━━━━━ 3s 729us/step - loss: 0.0013 - val_loss: 0.0013
Epoch 12/50
3747/37

INFO:tensorflow:Assets written to: /tmp/tmphv6x_zcx/assets


Saved artifact at '/tmp/tmphv6x_zcx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140406757575952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140406757579600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140406757578256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140406757576912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140406757580176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140406757577104: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/patrik/Diplomka/LODA/env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1786721557.600656    6298 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786721557.600667    6298 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 17:32:37.600958: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmphv6x_zcx
2026-08-14 17:32:37.601268: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 17:32:37.601271: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmphv6x_zcx
I0000 00:00:1786721557.603404    6298 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 17:32:37.603666: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 1

<Sequential name=sequential, built=True>